# DR-EOT Demo — Quick-Start

This notebook demonstrates the core workflow for **Density-Reweighted Entropic Optimal Transport (DR-EOT)**.

**Scenario:** two small synthetic datasets lying on 1-D manifolds with mismatched sampling densities. Standard EOT aligns them by density; DR-EOT recovers the geometrically correct correspondence.

---

## What is DR-EOT?

Standard EOT solves:
$$
\min_{W \geq 0} \sum_{ij} C_{ij} W_{ij} + \varepsilon \sum_{ij} W_{ij} \log W_{ij}
\quad \text{s.t.} \quad W\mathbf{1} = \mathbf{a},\; W^\top\mathbf{1} = \mathbf{b}
$$
with **uniform** marginals $\mathbf{a} = \frac{1}{m}\mathbf{1}_m$, $\mathbf{b} = \frac{1}{n}\mathbf{1}_n$. When sampling densities differ, the plan matches by density rather than geometry.

DR-EOT replaces the kernel $K_{ij} = e^{-C_{ij}/\varepsilon}$ with a density-adjusted kernel
$$M_{ij}^{(\theta)} = \frac{e^{-\|x_i - y_j\|^2/\varepsilon}}{\hat{f}_i^\theta \, \hat{g}_j^\theta}$$
and sets reweighted marginal constraints (see Algorithm 1 in the paper). The parameter $\theta \in [0,1]$ interpolates between standard EOT ($\theta=0$) and geometry-driven alignment ($\theta=1$).

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
from sklearn.metrics import pairwise_distances
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

# Core API
from dreot import sinkhorn_eot, sinkhorn_dreot, GlobalBootstrapLepski

## Step 1 — Generate two datasets with opposing sampling densities

In [ ]:
np.random.seed(0)
m, n = 2000, 3000

# Dataset X: points on y=x, sampled heavily on the left (x < 0.5)
x_X = np.concatenate([
    np.random.uniform(0, 0.5, int(0.9 * m)),
    np.random.uniform(0.5, 1, int(0.1 * m)),
])
X = np.column_stack([x_X, x_X])

# Dataset Y: points on y = 2 + x + 0.5x², sampled heavily on the right (x > 0.5)
x_Y = np.concatenate([
    np.random.uniform(0, 0.5, int(0.1 * n)),
    np.random.uniform(0.5, 1, int(0.9 * n)),
])
Y = np.column_stack([x_Y, 2 + x_Y + 0.5 * x_Y**2])

# True arc-length densities (used for DR-EOT with known densities)
mu_true = np.where(x_X < 0.5, 0.9 / 0.5, 0.1 / 0.5)       # piecewise-uniform pdf
mu_true /= np.sqrt(2)                                        # divide by |γ'| = sqrt(2)
nu_true = np.where(x_Y < 0.5, 0.1 / 0.5, 0.9 / 0.5)
nu_true /= np.sqrt(1 + (1 + x_Y)**2)

print(f"X: {X.shape}   Y: {Y.shape}")

## Step 2 — Estimate densities via Bootstrap-Lepski KDE  *(or use true densities)*

In practice densities are unknown and must be estimated. We use the Bootstrap-Lepski adaptive bandwidth selector. For this small demo the true densities (computed analytically above) can also be plugged in directly — see the commented block.

In [ ]:
from dreot.utils import knn_median

dist_XX = pairwise_distances(X, X, metric="sqeuclidean")
dist_YY = pairwise_distances(Y, Y, metric="sqeuclidean")

h_lo_X = 0.5 * np.sqrt(max(knn_median(dist_XX, 5),  1e-8))
h_hi_X = 0.5 * np.sqrt(knn_median(dist_XX, 500))
h_lo_Y = 0.5 * np.sqrt(max(knn_median(dist_YY, 5),  1e-8))
h_hi_Y = 0.5 * np.sqrt(knn_median(dist_YY, 500))

lepski_X = GlobalBootstrapLepski(X, (h_lo_X, h_hi_X), n_bandwidths=50,
                                  n_bootstrap=20, alpha=0.05, C=1.0, seed=42)
lepski_Y = GlobalBootstrapLepski(Y, (h_lo_Y, h_hi_Y), n_bandwidths=50,
                                  n_bootstrap=20, alpha=0.05, C=1.0, seed=42)

opt_X = lepski_X.select_bandwidth_global()
opt_Y = lepski_Y.select_bandwidth_global()

mu_est = opt_X.density_estimates   # shape (m,)
nu_est = opt_Y.density_estimates   # shape (n,)

print(f"Selected h_X = {opt_X.optimal_h:.4f}  h_Y = {opt_Y.optimal_h:.4f}")

## Step 3 — Compute transport plans

In [ ]:
eps = 5e-2
dist_XY = pairwise_distances(X, Y, metric="sqeuclidean")

# ── Standard EOT ──────────────────────────────────────────────────────────────
row_s, col_s = sinkhorn_eot(
    dist_XY, eps,
    np.ones((m, 1)) * n,
    np.ones((n, 1)) * m,
    delta=1e-6, max_iter=2000, check_freq=50, raise_on_bad_convergence=False,
)
W_eot = row_s * np.exp(-dist_XY / eps) * col_s.T

# ── DR-EOT with estimated densities, θ=1 ────────────────────────────────────
row_s_dr, col_s_dr = sinkhorn_dreot(
    dist_XY, eps,
    mu_est.reshape(-1, 1),
    nu_est.reshape(-1, 1),
    alpha=1,
    delta=1e-6, max_iter=2000, check_freq=50, raise_on_bad_convergence=False,
)
W_dreot = row_s_dr * np.exp(-dist_XY / eps) * col_s_dr.T

print(f"W_eot  : {W_eot.shape}")
print(f"W_dreot: {W_dreot.shape}")

## Step 4 — Visualize maximum-coupling correspondences

In [ ]:
vmin = min(mu_true.min(), nu_true.min())
vmax = max(mu_true.max(), nu_true.max())

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

for ax, W, title in zip(axes, [W_eot, W_dreot], ["Standard EOT", "DR-EOT (θ=1)"]):
    sc = ax.scatter(X[:, 0], X[:, 1], c=mu_true, cmap="viridis",
                    s=10, vmin=vmin, vmax=vmax, zorder=3)
    ax.scatter(Y[:, 0], Y[:, 1], c=nu_true, cmap="viridis",
               s=10, vmin=vmin, vmax=vmax, zorder=3)

    max_idx = np.argmax(W, axis=1)
    step = max(1, m // 50)
    for i in range(0, m, step):
        j = max_idx[i]
        ax.annotate("",
                    xy=(Y[j, 0], Y[j, 1]), xytext=(X[i, 0], X[i, 1]),
                    arrowprops=dict(arrowstyle="-|>", color="gray",
                                   lw=0.8, alpha=0.5, mutation_scale=6),
                    zorder=2)
    ax.set_title(title, fontsize=12)
    ax.set_xticks([]); ax.set_yticks([])

divider = make_axes_locatable(axes[1])
cax = divider.append_axes("right", size="4%", pad=0.05)
fig.colorbar(sc, cax=cax, label="Sampling Density")

plt.tight_layout()
plt.savefig("demo_result.pdf", bbox_inches="tight", dpi=150)
plt.show()
print("\nLeft:  EOT routes arrows toward the dense region (right side of Y).")
print("Right: DR-EOT recovers geometry-faithful correspondences.")

## Step 5 — Varying θ: controlling the density-geometry trade-off

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 2.5), sharey=False)

for ax, theta in zip(axes, [0.0, 0.33, 0.67, 1.0]):
    row_s, col_s = sinkhorn_dreot(
        dist_XY, eps,
        mu_est.reshape(-1, 1), nu_est.reshape(-1, 1),
        alpha=theta,
        delta=1e-6, max_iter=2000, check_freq=50, raise_on_bad_convergence=False,
    )
    W = row_s * np.exp(-dist_XY / eps) * col_s.T

    sc = ax.scatter(X[:, 0], X[:, 1], c=mu_true, cmap="viridis",
                    s=8, vmin=vmin, vmax=vmax, zorder=3)
    ax.scatter(Y[:, 0], Y[:, 1], c=nu_true, cmap="viridis",
               s=8, vmin=vmin, vmax=vmax, zorder=3)

    max_idx = np.argmax(W, axis=1)
    step = max(1, m // 40)
    for i in range(0, m, step):
        j = max_idx[i]
        ax.annotate("",
                    xy=(Y[j, 0], Y[j, 1]), xytext=(X[i, 0], X[i, 1]),
                    arrowprops=dict(arrowstyle="-|>", color="gray",
                                   lw=0.7, alpha=0.5, mutation_scale=5),
                    zorder=2)
    ax.set_title(rf"$\theta = {theta:.2f}$")
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()
print("As θ → 1, correspondences become increasingly geometry-driven.")